### Chatbot with message History


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

d:\GenAi\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026A6EC8FAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026A6ECF7980>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")])

AIMessage(content='Hello Kheman! 👋 Great to meet you. How’s engineering life treating you so far? What branch are you focusing on—mechanical, electrical, civil, computer, or something else? If you have any questions, need study tips, or just want to chat about your projects, I’m here to help!', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi My name is Kheman Saru and I am currently studying engineering". Probably they want a greeting, maybe ask about what they study, or how can I help. We should respond friendly, ask about their field, etc. No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 86, 'total_tokens': 222, 'completion_time': 0.291967651, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.003513926, 'prompt_tokens_details': None, 'queue_time': 0.157456781, 'total_time': 0.295481577}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [5]:
config = {"configurable":{"session_id":"chat_1"}}

In [6]:
response = with_message_history.invoke(
            [HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")],
            config=config
        )   

In [7]:
response.content

'Hello Kheman! 👋 Great to meet you. How’s engineering life treating you so far? Which branch are you studying, and what projects or topics have caught your interest lately? Feel free to share—I’m happy to chat about coursework, career ideas, or anything else on your mind!'

In [8]:
with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config
        )   

AIMessage(content='Your name is **Kheman\u202fSaru**.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" The conversation: they introduced themselves as Kheman Saru. So answer: Kheman Saru. Should be concise.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 161, 'total_tokens': 218, 'completion_time': 0.119731737, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.01022574, 'prompt_tokens_details': None, 'queue_time': 0.04563856, 'total_time': 0.129957477}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc365-27f1-7273-8e4d-ba7c1d6ae59a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 161, 'output_tokens': 57, 'total_tokens': 218, 'output_token_details': {'reasoning': 36}})

In [9]:
config1 = {"configurable":{"session_id":"chat_2"}}
response = with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config1
        )   
response.content

'I don’t actually know your name—unless you’ve shared it with me earlier in the conversation, I don’t have that information. If you’d like me to address you by name, feel free to let me know what it is!'

In [10]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system""You are an ai assistant.Answer all the query like a teacher with nice tone"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt|model


In [11]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is Kheman")]})


AIMessage(content='Hello, Kheman! 👋\n\nIt’s wonderful to meet you. I’m here to help you with any questions you have—whether it’s about a school subject, a personal project, or just something you’re curious about. \n\nWhat would you like to explore today? 🌟', additional_kwargs={'reasoning_content': 'We need to respond as a teacher with a nice tone. Greet and maybe ask about how can help. Also incorporate teacher-like approach.'}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 99, 'total_tokens': 196, 'completion_time': 0.205967987, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.00500733, 'prompt_tokens_details': None, 'queue_time': 0.04635904, 'total_time': 0.210975317}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc365-2c3f-7193-afbf-aa9f8c06b05e-0', tool_calls=[], invalid_tool_calls=[], 

In [12]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [13]:
config = {"configurable":{"session_id":"chat_4"}}
response=with_message_history.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)
response

AIMessage(content='I’m afraid I don’t know your name yet—our conversation hasn’t included that detail. Could you let me know what you’d like me to call you? Then I’ll be happy to address you by name in our future discussions!', additional_kwargs={'reasoning_content': 'The user says: "systemYou are an ai assistant.Answer all the query like a teacher with nice tone". Then asks "what is my name?". There\'s no prior context giving the name. As an AI, we don\'t know the user\'s name. We should respond politely, perhaps ask for clarification. Since we must answer like a teacher with nice tone. So say: "I don\'t know your name yet; could you tell me?" Also maybe explain we don\'t have that info.'}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 97, 'total_tokens': 251, 'completion_time': 0.317874273, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_time': 0.004377413, 'prompt_tokens_details': None, 'queue_time': 0.157851627, 'total_time': 0.3222516

In [15]:
from langchain_core.messages import trim_messages,SystemMessage,AIMessage
trimmer = trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="hi I am Kheman"),
    AIMessage(content="hi"),
    HumanMessage(content="I like vanilla icecream"),
    AIMessage(content="nice"),
    HumanMessage(content="what's 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks"),
    AIMessage(content="No problem"),
    HumanMessage(content="having Fun"),
    AIMessage(content="yes")
]
trimmer.invoke(messages)

[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi I am Kheman', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla icecream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="what's 2+2", additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No problem', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having Fun', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes', additional_kwargs={

In [19]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain = (RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
         |prompt
         |model
         
         )
response = chain.invoke(
    {
                "messages":messages + [HumanMessage(content="Which icecream do i like?")],
                "language":"English"
    }
            )
response.content

'You told me earlier that you enjoy **vanilla ice cream**—that’s the flavor you like! If you ever want to try something new, feel free to ask for recommendations. 🍦'